# Model Comparison Dashboard

Pulls all model-level MLflow runs from the experiment and produces a side-by-side comparison of test metrics, OOF metrics, and performance vs the dummy baseline.

**Run structure expected:**
- Root run: `Pipeline Run <timestamp>` — one per pipeline execution
- Child run: one per model (e.g. `xgboost`, `lightgbm`) — metrics logged here
- Grandchild runs: Optuna trials (filtered out)

In [ ]:
import mlflow
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

EXPERIMENT_NAME = "XGboost Experiments"
TRACKING_URI = "../mlruns"
PLOTS_DIR = Path("../artefacts/evaluation_plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

mlflow.set_tracking_uri(TRACKING_URI)

PALETTE = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52",
    "#8172B2", "#937860", "#DA8BC3",
]

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "font.size": 11,
})

## 1. Fetch runs from MLflow

In [ ]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment is None:
    raise RuntimeError(
        f"Experiment '{EXPERIMENT_NAME}' not found. "
        "Run the pipeline at least once before using this notebook."
    )

all_runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    run_view_type=mlflow.entities.ViewType.ACTIVE_ONLY,
)

print(f"Total runs found: {len(all_runs)}")

In [ ]:
# Model-level runs are:
#   1. Children of a root run (tags.mlflow.parentRunId is set)
#   2. Have auc_roc logged (filters out Optuna trial grandchildren)

is_child = all_runs["tags.mlflow.parentRunId"].notna()
has_metrics = all_runs["metrics.auc_roc"].notna()

model_runs = all_runs[is_child & has_metrics].copy()
model_runs["model"] = model_runs["tags.mlflow.runName"]

print(f"Model-level runs: {len(model_runs)}")
print("Models found:", model_runs["model"].unique().tolist())

## 2. Build comparison table

For each model, take the **most recent** run (identified by `start_time`). This means re-running the pipeline updates the comparison automatically.

In [ ]:
METRICS = [
    "auc_roc", "pr_auc", "brier_score",
    "f1", "precision", "recall", "accuracy",
]
OOF_METRICS = [f"oof_{m}" for m in METRICS]
DUMMY_METRICS = [f"dummy_{m}" for m in METRICS]

wanted_cols = ["model", "start_time"] + \
    [f"metrics.{m}" for m in METRICS + OOF_METRICS + DUMMY_METRICS]

available = [c for c in wanted_cols if c in model_runs.columns]
df = model_runs[available].copy()

# Keep the latest run per model
df = (
    df.sort_values("start_time", ascending=False)
    .groupby("model", as_index=False)
    .first()
    .set_index("model")
    .drop(columns="start_time")
)

# Strip the 'metrics.' prefix for readability
df.columns = [c.replace("metrics.", "") for c in df.columns]
df = df.sort_index()
df

In [ ]:
# Styled summary — highlight best per column (lower is better for brier_score)
test_cols = [m for m in METRICS if m in df.columns]

def highlight_best(s):
    """Green = best, red = worst. Brier score is inverted."""
    is_lower_better = "brier" in s.name
    if is_lower_better:
        best, worst = s.min(), s.max()
    else:
        best, worst = s.max(), s.min()
    return [
        "background-color: #c6efce" if v == best
        else "background-color: #ffc7ce" if v == worst
        else ""
        for v in s
    ]

summary = df[test_cols].round(4)
if len(summary) > 1:
    display(summary.style.apply(highlight_best, axis=0).format("{:.4f}"))
else:
    display(summary)

## 3. Test metrics comparison

In [ ]:
plot_metrics = [m for m in ["auc_roc", "pr_auc", "f1", "brier_score"] if m in df.columns]
n = len(plot_metrics)
models = df.index.tolist()
colors = PALETTE[:len(models)]

fig, axes = plt.subplots(1, n, figsize=(4 * n, 4), sharey=False)
if n == 1:
    axes = [axes]

for ax, metric in zip(axes, plot_metrics):
    vals = df[metric].values
    bars = ax.bar(models, vals, color=colors, width=0.55, edgecolor="white", linewidth=0.8)
    ax.set_title(metric.replace("_", " ").upper(), fontsize=10, fontweight="bold")
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models, rotation=30, ha="right", fontsize=9)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))
    lo = max(0, min(vals) - 0.05)
    hi = min(1, max(vals) + 0.05)
    ax.set_ylim(lo, hi)
    for bar, val in zip(bars, vals):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + (hi - lo) * 0.01,
            f"{val:.3f}",
            ha="center", va="bottom", fontsize=8,
        )

fig.suptitle("Test Set Metrics by Model", fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(PLOTS_DIR / "model_comparison_test_metrics.png", dpi=120, bbox_inches="tight")
plt.show()

## 4. Generalisation gap — test vs OOF

A large gap between OOF and test AUC-ROC indicates overfitting. A small gap confirms the CV estimate was reliable.

In [ ]:
gap_metrics = [(m, f"oof_{m}") for m in ["auc_roc", "pr_auc", "f1"] if m in df.columns and f"oof_{m}" in df.columns]

if not gap_metrics:
    print("No OOF metrics found — skipping generalisation gap chart.")
else:
    n = len(gap_metrics)
    fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4.5))
    if n == 1:
        axes = [axes]

    x = np.arange(len(models))
    w = 0.35

    for ax, (test_m, oof_m) in zip(axes, gap_metrics):
        test_vals = df[test_m].values
        oof_vals = df[oof_m].values

        ax.bar(x - w/2, oof_vals, width=w, label="OOF (CV)", color="#4C72B0", alpha=0.85, edgecolor="white")
        ax.bar(x + w/2, test_vals, width=w, label="Test", color="#DD8452", alpha=0.85, edgecolor="white")

        ax.set_xticks(x)
        ax.set_xticklabels(models, rotation=30, ha="right", fontsize=9)
        ax.set_title(test_m.replace("_", " ").upper(), fontsize=10, fontweight="bold")
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

        all_vals = np.concatenate([test_vals, oof_vals])
        lo = max(0, min(all_vals) - 0.04)
        hi = min(1, max(all_vals) + 0.04)
        ax.set_ylim(lo, hi)
        ax.legend(fontsize=9)

    fig.suptitle("Generalisation Gap: OOF vs Test", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "model_comparison_generalisation_gap.png", dpi=120, bbox_inches="tight")
    plt.show()

## 5. Performance vs dummy baseline

Shows how much each model improves over a stratified dummy classifier — the minimum bar any model must clear.

In [ ]:
baseline_pairs = [
    (m, f"dummy_{m}")
    for m in ["auc_roc", "pr_auc", "f1"]
    if m in df.columns and f"dummy_{m}" in df.columns
]

if not baseline_pairs:
    print("No dummy baseline metrics found — skipping baseline comparison.")
else:
    n = len(baseline_pairs)
    fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4.5))
    if n == 1:
        axes = [axes]

    x = np.arange(len(models))
    w = 0.35

    for ax, (model_m, dummy_m) in zip(axes, baseline_pairs):
        model_vals = df[model_m].values
        dummy_vals = df[dummy_m].values

        ax.bar(x - w/2, model_vals, width=w, label="Model", color="#55A868", alpha=0.85, edgecolor="white")
        ax.bar(x + w/2, dummy_vals, width=w, label="Dummy baseline", color="#aaaaaa", alpha=0.85, edgecolor="white")

        ax.set_xticks(x)
        ax.set_xticklabels(models, rotation=30, ha="right", fontsize=9)
        ax.set_title(model_m.replace("_", " ").upper(), fontsize=10, fontweight="bold")
        ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.3f"))

        all_vals = np.concatenate([model_vals, dummy_vals])
        lo = max(0, min(all_vals) - 0.04)
        hi = min(1, max(all_vals) + 0.06)
        ax.set_ylim(lo, hi)
        ax.legend(fontsize=9)

        # Annotate lift over baseline
        for xi, (mv, dv) in enumerate(zip(model_vals, dummy_vals)):
            lift = mv - dv
            ax.annotate(
                f"+{lift:.3f}",
                xy=(xi - w/2, mv),
                xytext=(xi - w/2, mv + (hi - lo) * 0.015),
                ha="center", fontsize=8, color="#2d6a2d",
            )

    fig.suptitle("Model vs Dummy Baseline", fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / "model_comparison_vs_baseline.png", dpi=120, bbox_inches="tight")
    plt.show()

## 6. Full metrics table

All logged metrics — test, OOF, and dummy — in one place.

In [ ]:
all_available = [m for m in METRICS + OOF_METRICS + DUMMY_METRICS if m in df.columns]

full_table = df[all_available].round(4).T

# Group rows under a MultiIndex for readability
def _group(col):
    if col.startswith("oof_"):
        return "OOF (CV)", col.replace("oof_", "")
    if col.startswith("dummy_"):
        return "Dummy baseline", col.replace("dummy_", "")
    return "Test set", col

full_table.index = pd.MultiIndex.from_tuples(
    [_group(c) for c in full_table.index],
    names=["split", "metric"],
)
display(full_table)